## **2. Inference**

In [1]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
import textwrap
import pandas as pd
import os
import re

seed = 42
with open(f"results_inferences_{seed}.pkl", "rb") as f:
    results_inferences = pickle.load(f)

# Output directory for per-dataset figures
out_dir = "inference_plots"
os.makedirs(out_dir, exist_ok=True)

alldatasets = []
datasets = list(results_inferences.keys())
n_datasets = len(datasets)

def _sanitize_filename(name: str) -> str:
    """Create a filesystem-safe filename fragment from a dataset name."""
    name = str(name)
    name = name.strip()
    # Replace spaces with underscores and remove characters that are problematic in filenames
    name = re.sub(r"[\s/\\]+", "_", name)
    name = re.sub(r"[^A-Za-z0-9_.\-()]", "", name)
    return name

# Iterate over datasets and produce one figure per dataset (saves and closes each figure to free memory)
for i, dataset_name in enumerate(datasets):
    print(f"[{i+1}/{n_datasets}] Plotting dataset: {dataset_name}")
    results = results_inferences[dataset_name]

    # Build combined dataframe for this dataset (same behavior as original code)
    _results = []
    for res in results:
        # res is expected to be a tuple (label, object) where object has a __dict__ and .risk().value
        try:
            df = pd.DataFrame(res[1].__dict__).drop(index=1, axis=0)
        except Exception:
            # If drop(index=1) fails (maybe index doesn't exist), try safer approach
            try:
                df = pd.DataFrame(res[1].__dict__)
            except Exception:
                # As a last resort, create a minimal DataFrame
                df = pd.DataFrame({"A": []})
        df["Atribute"] = [res[0]]
        _results.append(df)

    if len(_results) == 0:
        print(f"  No results found for dataset {dataset_name}, skipping.")
        continue

    df_results = pd.concat(_results, axis=0, ignore_index=True)
    df_results["Dataset"] = dataset_name
    alldatasets.append(df_results)

    # Extract risk values and labels (columns)
    try:
        risks = [res[1].risk().value for res in results]
    except Exception as e:
        # If risk extraction fails, try to extract from DataFrame if present
        print(f"  Warning: failed to extract risks via .risk().value ({e}). Attempting fallback.")
        risks = []
        for res in results:
            try:
                # fallback: check if res[1] has attribute 'risk' or 'risk_value' or 'risk_val'
                r = getattr(res[1], "risk")()
                risks.append(r.value)
            except Exception:
                # try attribute 'risk_value' or columns in df_results
                if "risk" in df_results.columns:
                    risks.append(float(df_results.loc[df_results["Atribute"] == res[0], "risk"].iloc[0]))
                else:
                    risks.append(np.nan)

    columns = [res[0] for res in results]

    # Create single figure for this dataset
    fig_w, fig_h = 8, 5
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=200)

    # X positions and bar plot
    x = np.arange(len(columns))
    cmap_color = plt.cm.get_cmap('viridis')((i) / max(1, n_datasets - 1))
    bar_colors = [cmap_color] * len(columns)
    ax.bar(x, risks, alpha=0.85, edgecolor='black', color=bar_colors)

    # Title and ticks
    ax.set_title(f"{dataset_name}", fontsize=12)
    wrapped = ["\n".join(textwrap.wrap(lbl, 15)) for lbl in columns]  # wrap longer labels
    ax.set_xticks(x)
    ax.set_xticklabels(wrapped, rotation=45, ha='right', fontsize=4)

    # Y-axis label only for plots where it makes sense (keep consistent)
    ax.set_ylabel("Measured Inference Risk", fontsize=9)

    # Mean risk annotation
    # compute mean only from numeric values
    risks_arr = np.array(risks, dtype=float)
    mean_risk = np.nanmean(risks_arr) if risks_arr.size > 0 else np.nan
    y_lim_max = (np.nanmax(risks_arr) * 1.1) if np.isfinite(np.nanmax(risks_arr)) else 1.0
    y_lim_max = y_lim_max if y_lim_max > 0 else 1.0
    ax.set_ylim(0, y_lim_max * 1.1)

    ax.text(0.95, 0.95,
            f'Mean Risk: {mean_risk:.4f}',
            transform=ax.transAxes,
            fontsize=9,
            color='red',
            fontweight='bold',
            ha='right', va='top',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', boxstyle='round,pad=0.3'))

    ax.grid(False)

    # Save and close figure to release memory
    safe_name = _sanitize_filename(dataset_name)
    outpath = os.path.join(out_dir, f"inference_risk_{safe_name}_{seed}.png")
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {outpath}")

# After all dataset figures are created, create and save the combined CSV as before
if len(alldatasets) > 0:
    df_InferencesRisk = pd.concat(alldatasets, axis=0, ignore_index=True)
    df_InferencesRisk.to_csv(f"df_InferencesRisk_{seed}.csv", index=False)
    print(f"Combined inference table saved to df_InferencesRisk_{seed}.csv")
else:
    print("No dataset results were collected; combined CSV not created.")

[1/5] Plotting dataset: avatarsk10_42


/tmp/ipykernel_2160357/310369020.py:86: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_color = plt.cm.get_cmap('viridis')((i) / max(1, n_datasets - 1))


  Saved: inference_plots/inference_risk_avatarsk10_42_42.png
[2/5] Plotting dataset: ctgan_42


/tmp/ipykernel_2160357/310369020.py:86: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_color = plt.cm.get_cmap('viridis')((i) / max(1, n_datasets - 1))


  Saved: inference_plots/inference_risk_ctgan_42_42.png
[3/5] Plotting dataset: gaussiancopula_42


...

  Saved: inference_plots/inference_risk_gaussiancopula_42_42.png
[4/5] Plotting dataset: synthpop_42


/tmp/ipykernel_2160357/310369020.py:86: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_color = plt.cm.get_cmap('viridis')((i) / max(1, n_datasets - 1))


  Saved: inference_plots/inference_risk_synthpop_42_42.png
[5/5] Plotting dataset: tvae_42


/tmp/ipykernel_2160357/310369020.py:86: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap_color = plt.cm.get_cmap('viridis')((i) / max(1, n_datasets - 1))


  Saved: inference_plots/inference_risk_tvae_42_42.png
Combined inference table saved to df_InferencesRisk_42.csv
